# 📦 Warehouse Restock Prediction

## Exploratory Data Analysis (EDA)

### Objective

The goal of this notebook is to understand the warehouse inventory datasets before building the production machine learning pipeline.

### Datasets

- Products.csv
- Warehouses.csv
- Sales_History.csv
- Inventory.csv

### Technology Stack

- Polars
- PyArrow
- Plotly
- NumPy

### Output

- Dataset understanding
- Data quality assessment
- Business insights
- Ready for feature engineering

In [70]:
from pathlib import Path

import warnings

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

import plotly.io as pio

pio.renderers.default = "browser"
warnings.filterwarnings("ignore")

In [71]:
from pathlib import Path

import warnings

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [72]:
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(50)
pl.Config.set_tbl_width_chars(150)

polars.config.Config

In [73]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"

PRODUCTS_PATH = RAW_DATA_DIR / "Products.csv"
WAREHOUSES_PATH = RAW_DATA_DIR / "Warehouses.csv"
SALES_PATH = RAW_DATA_DIR / "Sales_History.csv"
INVENTORY_PATH = RAW_DATA_DIR / "Inventory.csv"

In [74]:
files = [
    PRODUCTS_PATH,
    WAREHOUSES_PATH,
    SALES_PATH,
    INVENTORY_PATH,
]

for file in files:
    print(f"{file.name:<25} {'✅ Found' if file.exists() else '❌ Missing'}")

Products.csv              ✅ Found
Warehouses.csv            ✅ Found
Sales_History.csv         ✅ Found
Inventory.csv             ✅ Found


In [75]:
products_df = pl.read_csv(PRODUCTS_PATH)

warehouses_df = pl.read_csv(WAREHOUSES_PATH)

sales_df = pl.read_csv(SALES_PATH)

inventory_df = pl.read_csv(INVENTORY_PATH)

In [76]:
datasets = {
    "Products": products_df,
    "Warehouses": warehouses_df,
    "Sales": sales_df,
    "Inventory": inventory_df,
}

for name, df in datasets.items():
    print(f"{name:<15} Shape : {df.shape}")

Products        Shape : (500, 5)
Warehouses      Shape : (10, 3)
Sales           Shape : (180000, 11)
Inventory       Shape : (5000, 4)


In [77]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(f"{name.upper()} DATASET INFORMATION")
    print("=" * 80)

    print(f"Rows    : {df.height:,}")
    print(f"Columns : {df.width}")

    print("\nColumn Names")
    print("-" * 80)

    for column in df.columns:
        print(column)


PRODUCTS DATASET INFORMATION
Rows    : 500
Columns : 5

Column Names
--------------------------------------------------------------------------------
product_id
product_name
sku
category
brand

WAREHOUSES DATASET INFORMATION
Rows    : 10
Columns : 3

Column Names
--------------------------------------------------------------------------------
hub_id
hub_name
city

SALES DATASET INFORMATION
Rows    : 180,000
Columns : 11

Column Names
--------------------------------------------------------------------------------
year
month
hub_id
product_id
opening_stock
quantity_sold
promotion
discount_percentage
festival_flag
returns
closing_stock

INVENTORY DATASET INFORMATION
Rows    : 5,000
Columns : 4

Column Names
--------------------------------------------------------------------------------
hub_id
product_id
current_stock
last_updated


In [78]:
for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(f"{name.upper()} SCHEMA")
    print("=" * 80)

    schema = pl.DataFrame(
        {
            "Column": list(df.schema.keys()),
            "Data Type": [str(dtype) for dtype in df.schema.values()],
        }
    )

    display(schema)


PRODUCTS SCHEMA


Column,Data Type
str,str
"""product_id""","""String"""
"""product_name""","""String"""
"""sku""","""String"""
"""category""","""String"""
"""brand""","""String"""



WAREHOUSES SCHEMA


Column,Data Type
str,str
"""hub_id""","""String"""
"""hub_name""","""String"""
"""city""","""String"""



SALES SCHEMA


Column,Data Type
str,str
"""year""","""Int64"""
"""month""","""Int64"""
"""hub_id""","""String"""
"""product_id""","""String"""
"""opening_stock""","""Int64"""
…,…
"""promotion""","""Int64"""
"""discount_percentage""","""Int64"""
"""festival_flag""","""Int64"""



INVENTORY SCHEMA


Column,Data Type
str,str
"""hub_id""","""String"""
"""product_id""","""String"""
"""current_stock""","""Int64"""
"""last_updated""","""String"""


In [79]:
for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(f"{name.upper()} MISSING VALUES")
    print("=" * 80)

    missing = df.null_count().transpose(
        include_header=True,
        header_name="Column",
        column_names=["Missing Values"],
    )

    missing = missing.with_columns(
        (
            pl.col("Missing Values")
            / df.height
            * 100
        ).round(2).alias("Missing %")
    )

    display(missing)


PRODUCTS MISSING VALUES


Column,Missing Values,Missing %
str,u32,f64
"""product_id""",0,0.0
"""product_name""",0,0.0
"""sku""",0,0.0
"""category""",0,0.0
"""brand""",0,0.0



WAREHOUSES MISSING VALUES


Column,Missing Values,Missing %
str,u32,f64
"""hub_id""",0,0.0
"""hub_name""",0,0.0
"""city""",0,0.0



SALES MISSING VALUES


Column,Missing Values,Missing %
str,u32,f64
"""year""",0,0.0
"""month""",0,0.0
"""hub_id""",0,0.0
"""product_id""",0,0.0
"""opening_stock""",0,0.0
…,…,…
"""promotion""",0,0.0
"""discount_percentage""",0,0.0
"""festival_flag""",0,0.0



INVENTORY MISSING VALUES


Column,Missing Values,Missing %
str,u32,f64
"""hub_id""",0,0.0
"""product_id""",0,0.0
"""current_stock""",0,0.0
"""last_updated""",0,0.0


In [80]:
duplicate_summary = []

for name, df in datasets.items():

    duplicates = df.height - df.unique().height

    duplicate_summary.append(
        {
            "Dataset": name,
            "Rows": df.height,
            "Duplicate Rows": duplicates,
        }
    )

duplicate_df = pl.DataFrame(duplicate_summary)

duplicate_df

Dataset,Rows,Duplicate Rows
str,i64,i64
"""Products""",500,0
"""Warehouses""",10,0
"""Sales""",180000,0
"""Inventory""",5000,0


In [81]:
for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(f"{name.upper()} UNIQUE VALUES")
    print("=" * 80)

    unique_summary = []

    for column in df.columns:

        unique_summary.append(
            {
                "Column": column,
                "Unique Values": df[column].n_unique(),
            }
        )

    display(pl.DataFrame(unique_summary))


PRODUCTS UNIQUE VALUES


Column,Unique Values
str,i64
"""product_id""",500
"""product_name""",500
"""sku""",500
"""category""",8
"""brand""",10



WAREHOUSES UNIQUE VALUES


Column,Unique Values
str,i64
"""hub_id""",10
"""hub_name""",10
"""city""",10



SALES UNIQUE VALUES


Column,Unique Values
str,i64
"""year""",3
"""month""",12
"""hub_id""",10
"""product_id""",500
"""opening_stock""",721
…,…
"""promotion""",2
"""discount_percentage""",5
"""festival_flag""",2



INVENTORY UNIQUE VALUES


Column,Unique Values
str,i64
"""hub_id""",10
"""product_id""",500
"""current_stock""",665
"""last_updated""",1


In [82]:
for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(f"{name.upper()} DESCRIPTIVE STATISTICS")
    print("=" * 80)

    display(df.describe())


PRODUCTS DESCRIPTIVE STATISTICS


statistic,product_id,product_name,sku,category,brand
str,str,str,str,str,str
"""count""","""500""","""500""","""500""","""500""","""500"""
"""null_count""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null
"""std""",null,null,null,null,null
"""min""","""P000001""","""Adidas Beauty 112""","""SKU00001""","""Beauty""","""Adidas"""
"""25%""",null,null,null,null,null
"""50%""",null,null,null,null,null
"""75%""",null,null,null,null,null
"""max""","""P000500""","""Sony Toys 94""","""SKU00500""","""Toys""","""Sony"""



WAREHOUSES DESCRIPTIVE STATISTICS


statistic,hub_id,hub_name,city
str,str,str,str
"""count""","""10""","""10""","""10"""
"""null_count""","""0""","""0""","""0"""
"""mean""",null,null,null
"""std""",null,null,null
"""min""","""HUB001""","""Ahmedabad Hub""","""Ahmedabad"""
"""25%""",null,null,null
"""50%""",null,null,null
"""75%""",null,null,null
"""max""","""HUB010""","""Pune Hub""","""Pune"""



SALES DESCRIPTIVE STATISTICS


statistic,year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""",180000.0,180000.0,"""180000""","""180000""",180000.0,180000.0,180000.0,180000.0,180000.0,180000.0,180000.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",2024.0,6.5,null,null,452.51605,78.877911,0.180911,1.815917,0.666667,1.496817,447.303644
"""std""",0.816499,3.452062,null,null,167.802963,36.812393,0.384946,4.895315,0.471406,1.548316,165.36968
"""min""",2023.0,1.0,"""HUB001""","""P000001""",180.0,14.0,0.0,0.0,0.0,0.0,180.0
"""25%""",2023.0,4.0,null,null,314.0,49.0,0.0,0.0,0.0,0.0,310.0
"""50%""",2024.0,7.0,null,null,440.0,76.0,0.0,0.0,1.0,1.0,435.0
"""75%""",2025.0,9.0,null,null,575.0,105.0,0.0,0.0,1.0,2.0,569.0
"""max""",2025.0,12.0,"""HUB010""","""P000500""",900.0,226.0,1.0,20.0,1.0,10.0,879.0



INVENTORY DESCRIPTIVE STATISTICS


statistic,hub_id,product_id,current_stock,last_updated
str,str,str,f64,str
"""count""","""5000""","""5000""",5000.0,"""5000"""
"""null_count""","""0""","""0""",0.0,"""0"""
"""mean""",null,null,439.4464,null
"""std""",null,null,165.768464,null
"""min""","""HUB001""","""P000001""",180.0,"""2025-12-31"""
"""25%""",null,null,300.0,null
"""50%""",null,null,425.0,null
"""75%""",null,null,559.0,null
"""max""","""HUB010""","""P000500""",876.0,"""2025-12-31"""


In [83]:
quality_report = []

for name, df in datasets.items():

    quality_report.append(
        {
            "Dataset": name,
            "Rows": df.height,
            "Columns": df.width,
            "Duplicates": df.height - df.unique().height,
            "Total Missing": df.null_count().sum_horizontal().item(),
        }
    )

quality_df = pl.DataFrame(quality_report)

quality_df

Dataset,Rows,Columns,Duplicates,Total Missing
str,i64,i64,i64,i64
"""Products""",500,5,0,0
"""Warehouses""",10,3,0,0
"""Sales""",180000,11,0,0
"""Inventory""",5000,4,0,0


In [84]:
print("=" * 80)
print("PRODUCTS OVERVIEW")
print("=" * 80)

print(f"Total Products : {products_df.height:,}")

display(
    products_df.select(
        [
            pl.col("category").n_unique().alias("Unique Categories"),
            pl.col("brand").n_unique().alias("Unique Brands"),
        ]
    )
)

PRODUCTS OVERVIEW
Total Products : 500


Unique Categories,Unique Brands
u32,u32
8,10


In [85]:
category_distribution = (
    products_df
    .group_by("category")
    .len()
    .sort("len", descending=True)
    .rename({"len": "product_count"})
)

display(category_distribution)

category,product_count
str,u32
"""Books""",71
"""Sports""",71
"""Beauty""",63
"""Home""",61
"""Fashion""",61
"""Toys""",60
"""Electronics""",58
"""Grocery""",55


In [86]:
brand_distribution = (
    products_df
    .group_by("brand")
    .len()
    .sort("len", descending=True)
)

display(brand_distribution.head(20))

brand,len
str,u32
"""Samsung""",61
"""LG""",56
"""Nike""",54
"""Puma""",54
"""Dell""",51
"""Boat""",49
"""HP""",47
"""Sony""",45
"""Adidas""",45


In [87]:
print("=" * 80)
print("WAREHOUSE OVERVIEW")
print("=" * 80)

print(f"Total Warehouses : {warehouses_df.height}")

WAREHOUSE OVERVIEW
Total Warehouses : 10


In [88]:
products_df.columns

['product_id', 'product_name', 'sku', 'category', 'brand']

In [89]:
warehouses_df.columns

['hub_id', 'hub_name', 'city']

In [90]:
sales_df.columns

['year',
 'month',
 'hub_id',
 'product_id',
 'opening_stock',
 'quantity_sold',
 'promotion',
 'discount_percentage',
 'festival_flag',
 'returns',
 'closing_stock']

In [91]:
inventory_df.columns

['hub_id', 'product_id', 'current_stock', 'last_updated']

In [92]:
city_distribution = (
    warehouses_df
    .group_by("city")
    .len()
    .rename({"len": "warehouse_count"})
    .sort("warehouse_count", descending=True)
)

display(city_distribution)

city,warehouse_count
str,u32
"""Kolkata""",1
"""Mumbai""",1
"""Kochi""",1
"""Delhi""",1
"""Pune""",1
"""Coimbatore""",1
"""Ahmedabad""",1
"""Chennai""",1
"""Hyderabad""",1


In [93]:
inventory_df.describe()

statistic,hub_id,product_id,current_stock,last_updated
str,str,str,f64,str
"""count""","""5000""","""5000""",5000.0,"""5000"""
"""null_count""","""0""","""0""",0.0,"""0"""
"""mean""",null,null,439.4464,null
"""std""",null,null,165.768464,null
"""min""","""HUB001""","""P000001""",180.0,"""2025-12-31"""
"""25%""",null,null,300.0,null
"""50%""",null,null,425.0,null
"""75%""",null,null,559.0,null
"""max""","""HUB010""","""P000500""",876.0,"""2025-12-31"""


In [94]:
fig = px.histogram(
    inventory_df.to_pandas(),
    x="current_stock",
    nbins=40,
    title="Current Stock Distribution",
)

fig.show()

In [95]:
top_inventory = (
    inventory_df
    .sort("current_stock", descending=True)
    .head(20)
)

display(top_inventory)

hub_id,product_id,current_stock,last_updated
str,str,i64,str
"""HUB004""","""P000377""",876,"""2025-12-31"""
"""HUB003""","""P000121""",875,"""2025-12-31"""
"""HUB010""","""P000102""",865,"""2025-12-31"""
"""HUB010""","""P000038""",860,"""2025-12-31"""
"""HUB003""","""P000165""",855,"""2025-12-31"""
…,…,…,…
"""HUB002""","""P000090""",840,"""2025-12-31"""
"""HUB004""","""P000159""",839,"""2025-12-31"""
"""HUB005""","""P000163""",838,"""2025-12-31"""


In [96]:
print("=" * 80)
print("SALES OVERVIEW")
print("=" * 80)

print(f"Total Transactions : {sales_df.height:,}")

display(sales_df.describe())

SALES OVERVIEW
Total Transactions : 180,000


statistic,year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""",180000.0,180000.0,"""180000""","""180000""",180000.0,180000.0,180000.0,180000.0,180000.0,180000.0,180000.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",2024.0,6.5,null,null,452.51605,78.877911,0.180911,1.815917,0.666667,1.496817,447.303644
"""std""",0.816499,3.452062,null,null,167.802963,36.812393,0.384946,4.895315,0.471406,1.548316,165.36968
"""min""",2023.0,1.0,"""HUB001""","""P000001""",180.0,14.0,0.0,0.0,0.0,0.0,180.0
"""25%""",2023.0,4.0,null,null,314.0,49.0,0.0,0.0,0.0,0.0,310.0
"""50%""",2024.0,7.0,null,null,440.0,76.0,0.0,0.0,1.0,1.0,435.0
"""75%""",2025.0,9.0,null,null,575.0,105.0,0.0,0.0,1.0,2.0,569.0
"""max""",2025.0,12.0,"""HUB010""","""P000500""",900.0,226.0,1.0,20.0,1.0,10.0,879.0


In [98]:
sales_trend = (
    sales_df
    .group_by(["year", "month"])
    .agg(
        pl.col("quantity_sold").sum().alias("monthly_sales")
    )
    .sort(["year", "month"])
)

display(sales_trend)

year,month,monthly_sales
i64,i64,i64
2023,1,405149
2023,2,334834
2023,3,406435
2023,4,405957
2023,5,336006
…,…,…
2025,8,406290
2025,9,406162
2025,10,475864


In [100]:
fig.show(renderer="browser")

In [102]:
sales_trend_pd = sales_trend.to_pandas()

sales_trend_pd["year_month"] = (
    sales_trend_pd["year"].astype(str)
    + "-"
    + sales_trend_pd["month"].astype(str).str.zfill(2)
)

fig = px.line(
    sales_trend_pd,
    x="year_month",
    y="monthly_sales",
    markers=True,
    title="Monthly Sales Trend",
)

fig.show(renderer="browser")

In [104]:
sales_trend_pd = sales_trend.to_pandas()

sales_trend_pd["year_month"] = (
    sales_trend_pd["year"].astype(str)
    + "-"
    + sales_trend_pd["month"].astype(str).str.zfill(2)
)

fig = px.line(
    sales_trend_pd,
    x="year_month",
    y="monthly_sales",
    markers=True,
    title="Monthly Sales Trend",
)

fig.show(renderer="browser")

In [107]:
top_products = (
    sales_df
    .group_by("product_id")
    .agg(
        pl.col("quantity_sold").sum().alias("total_sales")
    )
    .sort("total_sales", descending=True)
    .head(20)
)

display(top_products)

product_id,total_sales
str,i64
"""P000137""",49449
"""P000327""",49444
"""P000165""",49095
"""P000177""",48666
"""P000371""",48571
…,…
"""P000219""",47674
"""P000009""",47573
"""P000348""",47535


In [108]:
sales_category = (
    sales_df
    .join(
        products_df,
        on="product_id",
        how="left",
    )
    .group_by("category")
    .agg(
        pl.col("quantity_sold").sum().alias("total_sales")
    )
    .sort("total_sales", descending=True)
)

display(sales_category)

category,total_sales
str,i64
"""Books""",1995681
"""Sports""",1878218
"""Fashion""",1868812
"""Beauty""",1806832
"""Toys""",1732304
"""Grocery""",1679578
"""Electronics""",1641154
"""Home""",1595445
